<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.3-ai-gateway-pattern/notebooks/exercises-10_3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.3: The AI Gateway Pattern

*Module 10 · AI System Architecture & Production Deployment*

> Every LLM call in your system should flow through one gateway: authentication, routing, rate limiting, cost tracking, and logging — in one place. This lesson builds a production AI Gateway with FastAPI that you can deploy to Cloud Run and put in front of any number of models.

---

## About this notebook

This notebook contains the **5 runnable code examples** from the published lesson `lesson-10.3.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Auth & API Key Management — Who's Calling? — `part1_auth.py`
2. Step 2 — Token-Aware Rate Limiter — Limit by Requests AND Tokens — `part2_rate_limiter.py`
3. Step 3 — Real-Time Cost Tracker — Know Every Penny Before It's Spent — `part3_cost_tracker.py`
4. Step 4 — Structured Request Logger — Trace Every Call — `part4_logger.py`
5. Step 5 — Complete Gateway — All Layers Composed — `gateway.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai fastapi requests pydantic


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Auth & API Key Management — Who's Calling?

Per-team API keys with scoped model access and spending limits.

**`part1_auth.py`** · _Block 1 of 5_

AUTH + API KEY MANAGEMENT — Per-team keys with model access control


In [ ]:
# =============================================
# AUTH + API KEY MANAGEMENT
# Per-team keys with model access control
# and spending limits.
# =============================================

from fastapi import FastAPI, HTTPException, Request, Depends
from pydantic import BaseModel, Field
from dataclasses import dataclass, field
import time, hashlib, json, uuid
from collections import defaultdict

@dataclass
class APIKeyConfig:
    """Configuration for one API key."""
    key: str
    team: str
    allowed_models: list[str]           # ["gemini-2.0-flash", "gemini-2.5-pro"]
    rate_limit_rpm: int = 60             # requests per minute
    daily_budget_usd: float = 10.0      # max daily spend
    active: bool = True

class AuthManager:
    """Manage API keys and team permissions."""
    
    def __init__(self):
        self.keys: dict[str, APIKeyConfig] = {}
    
    def register_key(self, team: str, models: list[str],
                     rpm: int = 60, budget: float = 10.0) -> str:
        """Create a new API key for a team."""
        key = f"nsk_live_{uuid.uuid4().hex[:16]}"
        self.keys[key] = APIKeyConfig(key=key, team=team, allowed_models=models,
                                       rate_limit_rpm=rpm, daily_budget_usd=budget)
        return key
    
    def authenticate(self, api_key: str) -> APIKeyConfig:
        """Validate key and return config. Raises 401 if invalid."""
        config = self.keys.get(api_key)
        if not config or not config.active:
            raise HTTPException(401, "Invalid or inactive API key")
        return config
    
    def check_model_access(self, config: APIKeyConfig, model: str):
        """Check if this key can access the requested model."""
        if model not in config.allowed_models:
            raise HTTPException(403, f"Key not authorized for model '{model}'. Allowed: {config.allowed_models}")

# ── Setup ──
auth = AuthManager()

# Register teams with different access levels
free_key = auth.register_key("free_tier", ["gemini-2.0-flash"], rpm=10, budget=1.0)
pro_key = auth.register_key("pro_team", ["gemini-2.0-flash", "gemini-2.5-pro"], rpm=60, budget=50.0)
internal_key = auth.register_key("internal", ["gemini-2.0-flash", "gemini-2.5-pro", "gemini-2.5-flash"], rpm=200, budget=500.0)

print("Registered API keys:")
print(f"  Free:     {free_key[:20]}... (Flash only, 10rpm, $1/day)")
print(f"  Pro:      {pro_key[:20]}... (Flash+Pro, 60rpm, $50/day)")
print(f"  Internal: {internal_key[:20]}... (All models, 200rpm, $500/day)")


> **What just happened?** AuthManager creates scoped API keys: free tier gets Flash only at 10 rpm and $1/day budget. Pro gets Flash + Pro at 60 rpm and $50/day. Internal gets all models at 200 rpm and $500/day. authenticate() validates the key, check_model_access() ensures the key can use the requested model. Every team gets exactly the access and budget they need — no more, no less.


### Step 2 · Token-Aware Rate Limiter — Limit by Requests AND Tokens

10 simple queries and 1 giant document analysis shouldn't count the same.

**`part2_rate_limiter.py`** · _Block 2 of 5_

TOKEN-AWARE RATE LIMITER — Dual limits: requests per minute AND


In [ ]:
# =============================================
# TOKEN-AWARE RATE LIMITER
# Dual limits: requests per minute AND
# tokens per minute. Sliding window.
# =============================================

class TokenAwareRateLimiter:
    """Rate limit by both request count AND token count."""
    
    def __init__(self):
        self.request_windows: dict[str, list[float]] = defaultdict(list)
        self.token_windows: dict[str, list[tuple]] = defaultdict(list)  # (timestamp, token_count)
    
    def check(self, key: str, rpm_limit: int,
             estimated_tokens: int = 0,
             tpm_limit: int = 100000) -> dict:
        """Check both request and token rate limits."""
        now = time.time()
        window = 60  # 1-minute sliding window
        cutoff = now - window
        
        # Clean expired entries
        self.request_windows[key] = [t for t in self.request_windows[key] if t > cutoff]
        self.token_windows[key] = [(t, c) for t, c in self.token_windows[key] if t > cutoff]
        
        # Check request count
        req_count = len(self.request_windows[key])
        if req_count >= rpm_limit:
            oldest = min(self.request_windows[key])
            retry_after = round(oldest + window - now, 1)
            return {"allowed": False, "reason": "request_limit",
                    "limit": rpm_limit, "current": req_count,
                    "retry_after_s": retry_after}
        
        # Check token count
        token_count = sum(c for _, c in self.token_windows[key])
        if token_count + estimated_tokens > tpm_limit:
            return {"allowed": False, "reason": "token_limit",
                    "limit": tpm_limit, "current": token_count,
                    "retry_after_s": 10}
        
        # Record this request
        self.request_windows[key].append(now)
        if estimated_tokens > 0:
            self.token_windows[key].append((now, estimated_tokens))
        
        return {"allowed": True,
                "requests_remaining": rpm_limit - req_count - 1,
                "tokens_remaining": tpm_limit - token_count - estimated_tokens}

# ── Demo ──
limiter = TokenAwareRateLimiter()

print("Token-aware rate limiting:\n")
tests = [
    ("free_user", 10, 500, 10000),    # 10 rpm, 500 tokens, 10K tpm
    ("free_user", 10, 500, 10000),
    ("pro_user",  60, 8000, 100000), # 60 rpm, 8K tokens (big doc), 100K tpm
]

for key, rpm, tokens, tpm in tests:
    result = limiter.check(key, rpm, tokens, tpm)
    icon = "✅" if result["allowed"] else "🚫"
    print(f"  {icon} {key:12s} {tokens:5d} tokens → req_remaining={result.get('requests_remaining','—')} tok_remaining={result.get('tokens_remaining','—')}")


> **What just happened?** TokenAwareRateLimiter enforces two limits simultaneously: requests per minute (RPM) AND tokens per minute (TPM). A free user at 10 RPM can't bypass the limit by sending 10 requests with 50,000 tokens each — the TPM limit catches that. Both use a sliding window. Response includes retry_after_s so clients know when to try again. This is how OpenAI, Anthropic, and Google rate-limit their APIs — dual RPM + TPM limits.


### Step 3 · Real-Time Cost Tracker — Know Every Penny Before It's Spent

Track cost per request. Enforce daily budgets. Alert before they're exceeded.

**`part3_cost_tracker.py`** · _Block 3 of 5_

REAL-TIME COST TRACKER — Per-request cost calculation. Per-team daily


In [ ]:
# =============================================
# REAL-TIME COST TRACKER
# Per-request cost calculation. Per-team daily
# budget enforcement. Spend alerts.
# =============================================

from datetime import datetime, date

class GatewayCostTracker:
    """Track and enforce per-team LLM spending."""
    
    PRICING = {  # per 1M tokens (input, output)
        "gemini-2.0-flash":  (0.10, 0.40),
        "gemini-2.5-flash":  (0.15, 0.60),
        "gemini-2.5-pro":    (1.25, 10.0),
    }
    
    def __init__(self):
        self.daily_spend: dict[str, dict] = defaultdict(
            lambda: {"date": date.today().isoformat(), "usd": 0.0, "requests": 0}
        )
    
    def estimate_cost(self, model: str, input_tokens: int, output_tokens: int) -> float:
        """Estimate cost for a request."""
        prices = self.PRICING.get(model, (0.10, 0.40))
        return (input_tokens * prices[0] + output_tokens * prices[1]) / 1_000_000
    
    def check_budget(self, team: str, daily_limit: float) -> dict:
        """Check if team is within budget."""
        spend = self.daily_spend[team]
        # Reset if new day
        if spend["date"] != date.today().isoformat():
            self.daily_spend[team] = {"date": date.today().isoformat(), "usd": 0.0, "requests": 0}
            spend = self.daily_spend[team]
        
        remaining = daily_limit - spend["usd"]
        return {
            "within_budget": remaining > 0,
            "spent_usd": round(spend["usd"], 4),
            "remaining_usd": round(remaining, 4),
            "pct_used": round(spend["usd"] / daily_limit * 100, 1) if daily_limit > 0 else 0,
            "requests_today": spend["requests"],
        }
    
    def record(self, team: str, model: str, input_tokens: int, output_tokens: int) -> float:
        """Record a completed request's cost."""
        cost = self.estimate_cost(model, input_tokens, output_tokens)
        self.daily_spend[team]["usd"] += cost
        self.daily_spend[team]["requests"] += 1
        return cost

# ── Demo ──
tracker = GatewayCostTracker()

print("Cost tracking:\n")
for model, in_tok, out_tok in [("gemini-2.0-flash",2000,500), ("gemini-2.5-pro",5000,2000)]:
    cost = tracker.record("pro_team", model, in_tok, out_tok)
    print(f"  {model:22s} {in_tok:5d}in + {out_tok:4d}out = ${cost:.6f}")

budget = tracker.check_budget("pro_team", 50.0)
print(f"\n  Budget: ${budget['spent_usd']} / $50.00 ({budget['pct_used']}% used, {budget['requests_today']} requests)")


> **What just happened?** GatewayCostTracker calculates real costs using actual API pricing for each model. Per-team daily budgets reset at midnight. check_budget() runs BEFORE each request — if the team has exceeded their daily limit, the request is blocked. Flash (2K in + 500 out) ≈ $0.0004. Pro (5K in + 2K out) ≈ $0.026 — 65x more expensive for the same token count. This is why the router from 10.1 matters.


### Step 4 · Structured Request Logger — Trace Every Call

Every request: who called, what model, how long, how many tokens, what it cost.

**`part4_logger.py`** · _Block 4 of 5_

STRUCTURED REQUEST LOGGER — JSON logs compatible with Cloud Logging,


In [ ]:
# =============================================
# STRUCTURED REQUEST LOGGER
# JSON logs compatible with Cloud Logging,
# Datadog, Grafana, and any log aggregator.
# =============================================

class GatewayLogger:
    """Structured logging for every gateway request."""
    
    def __init__(self, service: str = "ai-gateway"):
        self.service = service
    
    def log_request(self, *,
                    request_id: str,
                    team: str,
                    model: str,
                    input_tokens: int,
                    output_tokens: int,
                    latency_ms: float,
                    cost_usd: float,
                    status: str = "success",
                    cached: bool = False,
                    error: str = ""):
        """Log a completed request in structured JSON."""
        entry = {
            "severity": "INFO" if status == "success" else "ERROR",
            "timestamp": datetime.now().isoformat(),
            "service": self.service,
            "request_id": request_id,
            "team": team,
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
            "latency_ms": round(latency_ms, 1),
            "cost_usd": round(cost_usd, 6),
            "status": status,
            "cached": cached,
        }
        if error:
            entry["error"] = error[:200]
        
        print(json.dumps(entry))
        return entry

# ── Demo ──
logger = GatewayLogger()

logger.log_request(
    request_id="req-a1b2c3", team="pro_team", model="gemini-2.0-flash",
    input_tokens=1500, output_tokens=400, latency_ms=845, cost_usd=0.00031,
)


> **What just happened?** GatewayLogger produces structured JSON logs that Cloud Logging, Datadog, and Grafana can parse automatically. Every log entry includes: request_id (for tracing), team, model, token counts, latency, cost, status, and cache hit. This enables dashboards that answer: "Which team is spending the most?", "Which model has the highest latency?", "What's our cache hit rate?"


### Step 5 · Complete Gateway — All Layers Composed

Auth → rate limit → budget check → route → execute → log → respond.

**`gateway.py`** · _Block 5 of 5_

COMPLETE AI GATEWAY — Auth → Rate Limit → Budget → Route → Log.


In [ ]:
# =============================================
# COMPLETE AI GATEWAY
# Auth → Rate Limit → Budget → Route → Log.
# Deploy to Cloud Run as a single service.
# =============================================

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY", ""))

app = FastAPI(title="Netsetos AI Gateway", version="1.0.0")

# ── Initialize all layers ──
auth = AuthManager()
rate_limiter = TokenAwareRateLimiter()
cost_tracker = GatewayCostTracker()
gw_logger = GatewayLogger()

# ── Register default keys ──
DEMO_KEY = auth.register_key("demo", ["gemini-2.0-flash"], rpm=30, budget=5.0)

# ── Request model ──
class GatewayRequest(BaseModel):
    model: str = "gemini-2.0-flash"
    messages: list[dict]
    temperature: float = 0.7
    max_tokens: int = 1000

# ── The gateway endpoint ──
@app.post("/v1/chat")
async def chat(req: GatewayRequest, request: Request):
    request_id = f"req-{uuid.uuid4().hex[:8]}"
    start = time.time()
    
    # ── Layer 1: AUTHENTICATE ──
    api_key = request.headers.get("X-API-Key", "")
    key_config = auth.authenticate(api_key)
    auth.check_model_access(key_config, req.model)
    
    # ── Layer 2: RATE LIMIT ──
    est_tokens = sum(len(m.get("content","")) // 4 for m in req.messages) + req.max_tokens
    rate_check = rate_limiter.check(key_config.team, key_config.rate_limit_rpm, est_tokens)
    if not rate_check["allowed"]:
        raise HTTPException(429, detail={
            "error": "Rate limit exceeded",
            "reason": rate_check["reason"],
            "retry_after_s": rate_check["retry_after_s"],
        })
    
    # ── Layer 3: BUDGET CHECK ──
    budget = cost_tracker.check_budget(key_config.team, key_config.daily_budget_usd)
    if not budget["within_budget"]:
        raise HTTPException(402, detail={
            "error": "Daily budget exceeded",
            "spent": budget["spent_usd"],
            "limit": key_config.daily_budget_usd,
        })
    
    # ── Layer 4: EXECUTE ──
    try:
        model = genai.GenerativeModel(req.model,
            generation_config={"temperature": req.temperature, "max_output_tokens": req.max_tokens})
        prompt = "\n".join(m.get("content", "") for m in req.messages)
        response = model.generate_content(prompt)
        
        # Extract token counts from response metadata
        usage = getattr(response, "usage_metadata", None)
        in_tokens = getattr(usage, "prompt_token_count", len(prompt) // 4) if usage else len(prompt) // 4
        out_tokens = getattr(usage, "candidates_token_count", len(response.text) // 4) if usage else len(response.text) // 4
        
        # ── Layer 5: RECORD COST ──
        cost = cost_tracker.record(key_config.team, req.model, in_tokens, out_tokens)
        latency = round((time.time() - start) * 1000, 1)
        
        # ── Layer 6: LOG ──
        gw_logger.log_request(
            request_id=request_id, team=key_config.team, model=req.model,
            input_tokens=in_tokens, output_tokens=out_tokens,
            latency_ms=latency, cost_usd=cost)
        
        return {
            "id": request_id,
            "text": response.text,
            "model": req.model,
            "usage": {"input_tokens": in_tokens, "output_tokens": out_tokens},
            "cost_usd": round(cost, 6),
            "latency_ms": latency,
        }
    
    except HTTPException:
        raise
    except Exception as e:
        latency = round((time.time() - start) * 1000, 1)
        gw_logger.log_request(
            request_id=request_id, team=key_config.team, model=req.model,
            input_tokens=0, output_tokens=0, latency_ms=latency,
            cost_usd=0, status="error", error=str(e))
        raise HTTPException(500, detail=f"LLM error: {str(e)[:100]}")

# ── Dashboard endpoint ──
@app.get("/v1/dashboard/{team}")
async def dashboard(team: str):
    """Team spending dashboard."""
    budget = cost_tracker.check_budget(team, 50.0)
    return {
        "team": team,
        "today": budget,
        "pricing": cost_tracker.PRICING,
    }

print("""
Deploy to Cloud Run:

  gcloud run deploy ai-gateway \\
    --source . \\
    --region asia-south1 \\
    --allow-unauthenticated \\
    --set-env-vars GOOGLE_API_KEY=your-key

Test with curl:

  curl -X POST https://ai-gateway-xxx.run.app/v1/chat \\
    -H "X-API-Key: YOUR_KEY" \\
    -H "Content-Type: application/json" \\
    -d '{"model":"gemini-2.0-flash","messages":[{"content":"Hello"}]}'
""")


> **What just happened?** The complete gateway pipeline in one endpoint: (1) Authenticate — validate API key, check model access. (2) Rate limit — check RPM + TPM, return 429 with retry_after if exceeded. (3) Budget check — return 402 if daily spend exceeded. (4) Execute — call the requested model via Gemini API. (5) Record cost — track actual tokens used and dollar cost. (6) Log — structured JSON with request_id, team, model, tokens, latency, cost. The response includes usage and cost so clients can track their own spending. A dashboard endpoint shows per-team daily spend.


---

## Wrap-up

You've walked through all 5 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
